In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
"""
Multilevel Aggregates
Multilevel aggregates allows summarizing data at several hierarchical levels.
We have three variations of multilevel aggregates in Spark.

    Rollup
    Cube
    Grouping Sets
"""

'\nMultilevel Aggregates\nMultilevel aggregates allows summarizing data at several hierarchical levels.\nWe have three variations of multilevel aggregates in Spark.\n\n    Rollup\n    Cube\n    Grouping Sets\n'

In [ ]:
members_df = spark.table("spark_db.members")
bookings_df = spark.table("spark_db.bookings")
facilities_df = spark.table("spark_db.facilities")

club_bookings_df = (
    bookings_df.join(facilities_df, "facid")
            .join(members_df, "memid", "left")
            .selectExpr("bookid",
                        "case when memid==0 then 'Guest Member' else concat_ws(' ', firstname, surname) end as member_name",
                        "fac_name","starttime",
                        "case when memid == 0 then slots * guestcost else slots * membercost end as booking_amount")            
)

club_bookings_df.show()

+------+------------+---------------+-------------------+--------------+
|bookid| member_name|       fac_name|          starttime|booking_amount|
+------+------------+---------------+-------------------+--------------+
|     0|Darren Smith|   Table Tennis|2022-07-03 11:00:00|             0|
|     1|Darren Smith| Massage Room 1|2022-07-03 08:00:00|            70|
|     2|Guest Member|   Squash Court|2022-07-03 18:00:00|          NULL|
|     3|Darren Smith|  Snooker Table|2022-07-03 19:00:00|             0|
|     4|Darren Smith|     Pool Table|2022-07-03 10:00:00|             0|
|     5|Darren Smith|     Pool Table|2022-07-03 15:00:00|             0|
|     6| Tracy Smith| Tennis Court 1|2022-07-04 09:00:00|            15|
|     7| Tracy Smith| Tennis Court 1|2022-07-04 15:00:00|            15|
|     8|  Tim Rownam| Massage Room 1|2022-07-04 13:30:00|            70|
|     9|Guest Member| Massage Room 1|2022-07-04 15:00:00|           160|
|    10|Guest Member| Massage Room 1|2022-07-04 17:

In [ ]:
"""
Q1. Prepare a monthly revenue report for year 2022.

Also roll up the total for the month column.

+----+--------+
|mnth| revenue|
+----+--------+
|   7| 23202.5|
|   8| 46066.5|
|   9| 63315.5|
|    |132584.5|
+----+--------+
"""
from pyspark.sql.functions import col, year, month, sum

result_df = club_bookings_df.filter(year(col("starttime")) == 2022)\
                            .withColumn("mnth", month(col("starttime")))\
                            .rollup(col("mnth"))\
                            .agg(sum(col("booking_amount")).alias("revenue"))\
                            .orderBy(col("mnth").asc_nulls_last()) # used rollup instead of group by to get the total across all months # rollup is an advanced groupBy
result_df.show()

+----+-------+
|mnth|revenue|
+----+-------+
|   7|  20800|
|   8|  40945|
|   9|  55465|
|NULL| 117210|
+----+-------+



In [ ]:
"""
Q2. Prepare a revenue report by revenue_from (Guest/Member) and facility_name for year 2022.

Also roll up the total for each group.

+------------+---------------+-------+
|revenue_from|  facility_name|revenue|
+------------+---------------+-------+
|       Guest|Badminton Court| 1906.5|
|       Guest| Massage Room 1|41600.0|
|       Guest| ..............|.......|
|       Guest|     ..........|  .....|
|       Guest|           NULL|89096.5|
|      Member|Badminton Court|    0.0|
|      Member| Massage Room 1|30940.0|
|      Member| ..............| ......|
|      Member| ..............| ......|
|      Member|           NULL|43488.0|
+------------+---------------+-------+
"""
from pyspark.sql.functions import expr

result_df = club_bookings_df.filter(year(col("starttime")) == 2022)\
                            .withColumn("revenue_from", expr("case when member_name=='Guest Member' then 'Guest' else 'Member' end"))\
                            .rollup(col("revenue_from"), col("fac_name").alias("facility_name"))\
                            .agg(sum(col("booking_amount")).alias("revenue"))\
                            .orderBy(col("revenue_from").asc_nulls_last(), col("facility_name").asc_nulls_last()) 
# here rollup will give results in 3 dimensions # where facility_name is not known but revenue_from is GUEST, second where it is from Member and last where both are null and the total across everything
result_df.show()

+------------+---------------+-------+
|revenue_from|  facility_name|revenue|
+------------+---------------+-------+
|       Guest|Badminton Court|   NULL|
|       Guest| Massage Room 1|  41600|
|       Guest| Massage Room 2|  13920|
|       Guest|     Pool Table|    270|
|       Guest|  Snooker Table|    240|
|       Guest|   Squash Court|   NULL|
|       Guest|   Table Tennis|    180|
|       Guest| Tennis Court 1|   9075|
|       Guest| Tennis Court 2|   9900|
|       Guest|           NULL|  75185|
|      Member|Badminton Court|      0|
|      Member| Massage Room 1|  30940|
|      Member| Massage Room 2|   1890|
|      Member|     Pool Table|      0|
|      Member|  Snooker Table|      0|
|      Member|   Squash Court|   NULL|
|      Member|   Table Tennis|      0|
|      Member| Tennis Court 1|   4785|
|      Member| Tennis Court 2|   4410|
|      Member|           NULL|  42025|
+------------+---------------+-------+
only showing top 20 rows



In [ ]:
"""
Q3. Prepare a revenue report by revenue_from(Guest/Member) and facility_name for year 2022.

Also compute totals for all 4 dimensions of revenue_from and facility_name.

    (revenue_from, facility_name) : rollup
    (revenue_from, ) : rollup
    (facility_name, ) : not available in roolup
    ( , ) : grand total in rollup
"""
from pyspark.sql.functions import expr

result_df = club_bookings_df.filter(year(col("starttime")) == 2022)\
                            .withColumn("revenue_from", expr("case when member_name=='Guest Member' then 'Guest' else 'Member' end"))\
                            .cube(col("revenue_from"), col("fac_name").alias("facility_name"))\
                            .agg(sum(col("booking_amount")).alias("revenue"))\
                            .orderBy(col("revenue_from").asc_nulls_last(), col("facility_name").asc_nulls_last())
# here we have used cube instead of rollup and groupBy. # this will give us 4 dimenstions mentioned in the question. # 1 dimension more than rollup in this case but could be more in other cases where columns used in group are more 
result_df.show()

+------------+---------------+-------+
|revenue_from|  facility_name|revenue|
+------------+---------------+-------+
|       Guest|Badminton Court|   NULL|
|       Guest| Massage Room 1|  41600|
|       Guest| Massage Room 2|  13920|
|       Guest|     Pool Table|    270|
|       Guest|  Snooker Table|    240|
|       Guest|   Squash Court|   NULL|
|       Guest|   Table Tennis|    180|
|       Guest| Tennis Court 1|   9075|
|       Guest| Tennis Court 2|   9900|
|       Guest|           NULL|  75185|
|      Member|Badminton Court|      0|
|      Member| Massage Room 1|  30940|
|      Member| Massage Room 2|   1890|
|      Member|     Pool Table|      0|
|      Member|  Snooker Table|      0|
|      Member|   Squash Court|   NULL|
|      Member|   Table Tennis|      0|
|      Member| Tennis Court 1|   4785|
|      Member| Tennis Court 2|   4410|
|      Member|           NULL|  42025|
+------------+---------------+-------+
only showing top 20 rows



In [ ]:
"""
Q4: Prepare a revenue report similar to the following.

  revenue_from  | facility_name   | revenue
  ---------------------------------------
  Guest         | Badminton Court | 1906.5
  Guest         | Massage Room 1  | 41600
  Guest         | Massage Room 2  | 13920
  Guest         |                 | 57426.5
  Member        | Badminton Court | 0
  Member        | Massage Room 1  | 30940
  Member        | Massage Room 2  | 1890
  Member        |                 | 32830
"""
# rollup done only for facility_name
result_df = club_bookings_df.filter(year(col("starttime")) == 2022)\
                            .withColumn("revenue_from", expr("case when member_name=='Guest Member' then 'Guest' else 'Member' end"))\
                            .groupingSets([(col("revenue_from"), col("fac_name")), (col("revenue_from"), )], col("revenue_from"), col("fac_name").alias("facility_name"))\
                            .agg(sum(col("booking_amount")).alias("revenue"))\
                            .orderBy(col("revenue_from").asc_nulls_last(), col("facility_name").asc_nulls_last())
# this DF function does not exist in the version of spark I am using in local 
# in groupingSets function we can mention an array for the dimensions and that will be considered
# we mention multiples sets so we can aggregate on those. 
# groupBy columns are mandatory in groupingSets as that is a dimension as well
result_df.show()

AttributeError: 'DataFrame' object has no attribute 'groupingSets'